# Seminar 12: Fine-Tuning DistilBERT for Text Classification

**Complete Version**

Today is a practical supervised fine-tuning lab with more model interaction than boilerplate.

Goals:
- build a prediction helper and test a model before fine-tuning,
- assemble a DistilBERT fine-tuning pipeline,
- compare model behavior before and after training,
- inspect an attention map carefully,
- evaluate mistakes without turning the notebook into a metrics-only lab.


## Facilitation Notes

Suggested pacing:
- setup and quick data recap: 5 minutes,
- Exercise 1: 20 minutes,
- Exercise 2: 20 minutes,
- fine-tuning demo: 10-15 minutes of runtime,
- Exercise 3: 15 minutes,
- Exercise 4: 15 minutes,
- Exercise 5 and wrap-up: 15 minutes.

Attention-map framing matters:
- it can show where one token reads information from in one layer/head,
- it is not a full explanation of the final class prediction,
- comparing heads/layers is often more honest than showing one pretty heatmap.


## 0. Setup

If you run this in Colab and packages are missing, uncomment the install line first.


In [ ]:
# If needed in Colab, uncomment:
# %pip install transformers datasets scikit-learn accelerate -q



In [ ]:
# If needed in Colab, uncomment:

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

pd.set_option('display.max_colwidth', 160)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


### Provided Data and Tokenizer Recap

Loading the dataset is setup, not a board exercise.

AG News has four labels:
- World,
- Sports,
- Business,
- Sci/Tech.

We also show a tiny tokenizer recap from Seminar 11, just enough to connect text classification to model inputs.


In [ ]:
CFG = {
    'model_name': 'distilbert-base-uncased',
    'max_length': 96,
    'train_size': 1600,
    'val_size': 400,
    'seed': 42,
    'num_epochs': 1,
    'batch_size': 16,
    'learning_rate': 2e-5,
}

raw_dataset = load_dataset('fancyzhx/ag_news')
label_names = raw_dataset['train'].features['label'].names

train_small = raw_dataset['train'].shuffle(seed=CFG['seed']).select(range(CFG['train_size']))
val_small = raw_dataset['test'].shuffle(seed=CFG['seed']).select(range(CFG['val_size']))

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

print('labels:', label_names)
print('train subset:', len(train_small))
print('validation subset:', len(val_small))

for i in range(3):
    label_id = train_small[i]['label']
    print('')
    print('label:', label_names[label_id])
    print(train_small[i]['text'])

sample_tokens = tokenizer.convert_ids_to_tokens(tokenizer(train_small[0]['text'])['input_ids'])
print('')
print('Tokenizer recap:')
print(sample_tokens[:20])


## 1. Exercise 1: Prediction Helper Before Fine-Tuning

Before fine-tuning, the DistilBERT base is pretrained, but the classification head is new for AG News. Its predictions should not be trusted yet.

Task:
- create a DistilBERT sequence-classification model,
- implement `predict_text`,
- test it on custom news-like examples before training,
- observe that the predictions are not meaningful yet.

Function contract:
- `predict_text(text, model, tokenizer, label_names)` receives one text string,
- it returns a table with columns `label`, `probability`, `rank`,
- probabilities should be sorted from largest to smallest.

Useful notes:
- `AutoModelForSequenceClassification.from_pretrained(..., num_labels=...)` creates the classifier head,
- `model(**encoded).logits` gives class logits with shape `[1, num_classes]`,
- `F.softmax(logits, dim=1)` converts logits to probabilities,
- `torch.topk(probabilities, k=...)` returns the largest probabilities and their label ids.


In [ ]:
custom_texts = [
    'The team scored two late goals to win the championship.',
    'The central bank raised interest rates again on Tuesday.',
    'Researchers announced a faster computer chip for mobile devices.',
    'Diplomats met to discuss the new peace agreement.',
]

id2label = {}
label2id = {}
for label_id in range(len(label_names)):
    label_name = label_names[label_id]
    id2label[label_id] = label_name
    label2id[label_name] = label_id

model = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'],
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
).to(device)


def predict_text(text, model, tokenizer, label_names):
    rows = []
    tokenize_text = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=CFG['max_length'],
    ).to(device)

    model.eval()
    with torch.no_grad():
      logits = model(**tokenize_text).logits
      probs = F.softmax(logits, dim=1)
      top = torch.topk(probs, k=len(label_names))
      for prob, idx in zip(top.values[0], top.indices[0]):
        # print(idx)
        # print(top)
        label = label_names[idx]
        rows.append({"label_name": label, "prob": prob, "rank": idx})
    return pd.DataFrame(rows)



pretrain_prediction_table = predict_text(custom_texts[0], model, tokenizer, label_names)
pretrain_prediction_table


### Checks (Exercise 1)

In [ ]:
# assert model is not None
# assert isinstance(pretrain_prediction_table, pd.DataFrame)
# for column in ['rank', 'label', 'probability']:
#     assert column in pretrain_prediction_table.columns

# assert len(pretrain_prediction_table) == len(label_names)
# assert pretrain_prediction_table['probability'].between(0, 1).all()

# probabilities = pretrain_prediction_table['probability'].tolist()
# for i in range(len(probabilities) - 1):
#     assert probabilities[i] >= probabilities[i + 1]

# print('Exercise 1 passed.')


## 2. Exercise 2: Training Pipeline Assembly

Now assemble the pieces that `Trainer` needs.

Task:
- implement `tokenize_batch`,
- tokenize train and validation subsets,
- implement `compute_metrics`,
- create the `Trainer` using the model from Exercise 1.

Function contracts:
- `tokenize_batch(batch)` receives a batch with a `text` field and returns padded tokenizer outputs,
- `compute_metrics(eval_pred)` receives `(logits, labels)` and returns `accuracy`, `precision`, `recall`, `f1`,
- logits have shape `[num_examples, num_classes]`.

Useful tokenizer arguments:
- `padding='max_length'` makes every example the same length inside the dataset,
- `truncation=True` cuts examples longer than `max_length`,
- `max_length=CFG['max_length']` sets the fixed length.

Useful functions:
- `np.argmax(logits, axis=-1)` converts logits to predicted class ids,
- `precision_recall_fscore_support(..., average='macro')` treats all classes equally.


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=CFG['max_length'],
    ).to(device)


tokenized_train = train_small.map(tokenize_batch, batched=True)
tokenized_val = val_small.map(tokenize_batch, batched=True)


def compute_metrics(eval_pred):
    metrics = {
        "accuracy": accuracy_score(eval_pred[1], np.argmax(eval_pred[0], axis=-1)),
        "precision": precision_recall_fscore_support(eval_pred[1], np.argmax(eval_pred[0], axis=-1), average="macro")[0],
        "recall": precision_recall_fscore_support(eval_pred[1], np.argmax(eval_pred[0], axis=-1), average="macro")[1],
        "f1": precision_recall_fscore_support(eval_pred[1], np.argmax(eval_pred[0], axis=-1), average="macro")[2]
      }
    return metrics


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='seminar12_distilbert_ag_news',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=CFG['learning_rate'],
    per_device_train_batch_size=CFG['batch_size'],
    per_device_eval_batch_size=CFG['batch_size'],
    num_train_epochs=CFG['num_epochs'],
    weight_decay=0.01,
    logging_steps=25,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Pipeline objects prepared.')


### Checks (Exercise 2)

In [ ]:
assert tokenized_train is not None
assert tokenized_val is not None

first_item = tokenized_train[0]
for key in ['input_ids', 'attention_mask', 'label']:
    assert key in first_item

sample_logits = np.array([
    [3.0, 1.0, 0.0, -1.0],
    [0.1, 0.2, 2.0, 0.0],
])
sample_labels = np.array([0, 2])
sample_metrics = compute_metrics((sample_logits, sample_labels))
for key in ['accuracy', 'precision', 'recall', 'f1']:
    assert key in sample_metrics

assert trainer is not None
print('Exercise 2 passed.')


## 3. Fine-Tuning Demo

Training is mostly waiting, so this is a run/demo cell rather than a separate exercise.

The model object from Exercise 1 is the same object used by the trainer. After training, `predict_text` should become much more meaningful.


In [ ]:
train_result = trainer.train()
validation_metrics = trainer.evaluate()
validation_metrics


## 4. Exercise 3: Prediction Playground After Fine-Tuning

Now interact with the fine-tuned model directly.

Task:
- reuse `predict_text`,
- run several custom texts,
- build a compact table with the top two labels and probabilities,
- include at least one ambiguous example.

Function contract:
- `build_playground_table(texts, model, tokenizer, label_names)` returns columns `text`, `top_label`, `top_probability`, `second_label`, `second_probability`.

Good examples to test:
- clear Sports,
- clear Business,
- clear Sci/Tech,
- clear World,
- ambiguous Business/SciTech headline.


In [ ]:
playground_texts = [
    'The striker scored in extra time as fans celebrated the final whistle.',
    'Shares of the company jumped after stronger quarterly earnings.',
    'The new smartphone processor improves battery life and AI performance.',
    'Government leaders agreed on a new trade deal after two days of talks.',
    'Apple announced record revenue after launching its latest chip.',
]


def build_playground_table(texts, model, tokenizer, label_names):
    rows = []

    for text in texts:
        prediction_table = predict_text(text, model, tokenizer, label_names)
        rows.append({
            'text': text,
            'top_label': prediction_table.loc[0, 'label'],
            'top_probability': prediction_table.loc[0, 'probability'],
            'second_label': prediction_table.loc[1, 'label'],
            'second_probability': prediction_table.loc[1, 'probability'],
        })

    return pd.DataFrame(rows)


playground_table = build_playground_table(playground_texts, model, tokenizer, label_names)
playground_table


### Checks (Exercise 3)

In [ ]:
assert isinstance(playground_table, pd.DataFrame)
required_columns = ['text', 'top_label', 'top_probability', 'second_label', 'second_probability']
for column in required_columns:
    assert column in playground_table.columns

assert len(playground_table) == len(playground_texts)
assert playground_table['top_probability'].between(0, 1).all()
assert playground_table['second_probability'].between(0, 1).all()

print('Exercise 3 passed.')


## 5. Exercise 4: Attention Map Inspection

Attention maps can be interesting, but be careful: attention is not a complete explanation of the final class decision.

What an attention map shows:

```text
for one layer and one head, how much each query token attends to each key token
```

Task:
- run one text with `output_attentions=True`,
- choose one layer and one head,
- plot attention from `[CLS]` to all tokens,
- optionally compare another head.

Function contract:
- `get_cls_attention(text, model, tokenizer, layer_index, head_index)` returns `tokens` and `attention_values`,
- `tokens` and `attention_values` must have the same length,
- attention values should sum to approximately 1.

Useful shape:
- `outputs.attentions[layer_index]` has shape `[batch, num_heads, sequence_length, sequence_length]`.


In [ ]:
attention_text = 'Apple announced record revenue after launching its latest chip.'


def get_attention_matrices(text, model, tokenizer, layer_index=0, head_index=0):
    model.eval()
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=CFG['max_length'],
    ).to(device)

    # model.config.output_attentions = True
    if hasattr(model.config, '_attn_implementation'):
        model.config._attn_implementation = 'eager'

    with torch.no_grad():
        outputs = model(**encoded, output_attentions=True, return_dict=True)

    if outputs.attentions is None:
        raise ValueError("No attention tensors were returned. Re-run the model creation cell so attn_implementation='eager' is used.")

    input_ids = encoded['input_ids'][0].detach().cpu()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    # layer_attention = outputs.attentions[layer_index]
    # head_attention = layer_attention[0, head_index]
    # attention_matrix = layer_attention.detach().cpu().numpy()

    return tokens, outputs.attentions


def plot_attention_matrix(tokens, attention_matrix, title):
    plt.figure(figsize=(8, 7))
    plt.imshow(attention_matrix.detach().cpu(), cmap='viridis', vmin=0.0, vmax=attention_matrix.max())
    plt.colorbar(label='attention weight')
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel('key token: attended to')
    plt.ylabel('query token: attending from')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_all_attention_layer(attention_tokens, attention_matrices, layer=0):
    for i in range(attention_matrices.shape[1]):
      plot_attention_matrix(attention_tokens, attention_matrices[0, i], f'Attention map: layer {layer}, head {i}')



attention_tokens, attention_matrices = get_attention_matrices(
    attention_text,
    model,
    tokenizer,
    layer_index=0,
    head_index=0,
)

plot_attention_matrix(attention_tokens, attention_matrices[0][0, 0], 'Attention map: layer 0, head 0')

plot_all_attention_layer(attention_tokens, attention_matrices[0])


### Checks (Exercise 4)

In [ ]:
assert isinstance(attention_tokens, list)
assert len(attention_tokens) > 0

print('Exercise 4 passed.')


## 6. Exercise 5: Minimal Evaluation and Confident Mistakes

We still need a real validation check, but keep it focused.

Task:
- build a prediction table for the validation subset,
- draw a confusion matrix,
- show the most confident mistakes.

Function contracts:
- `build_prediction_table(trainer, dataset, raw_dataset, label_names)` returns `text`, `true_label`, `predicted_label`, `confidence`, `correct`,
- `build_confident_mistakes(prediction_table, max_rows=10)` returns incorrect examples sorted by confidence descending.

Useful notes:
- `trainer.predict(dataset)` returns logits and labels,
- logits shape is `[N, num_classes]`,
- softmax converts logits to probabilities.


In [ ]:
def build_prediction_table(trainer, dataset, raw_dataset, label_names):
    prediction_output = trainer.predict(dataset)
    logits = prediction_output.predictions
    labels = prediction_output.label_ids

    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predicted_ids = np.argmax(logits, axis=-1)

    rows = []
    for i in range(len(labels)):
        predicted_id = int(predicted_ids[i])
        true_id = int(labels[i])
        confidence = float(probabilities[i, predicted_id])
        rows.append({
            'text': raw_dataset[i]['text'],
            'true_label': label_names[true_id],
            'predicted_label': label_names[predicted_id],
            'confidence': confidence,
            'correct': predicted_id == true_id,
        })

    return pd.DataFrame(rows)


def build_confident_mistakes(prediction_table, max_rows=10):
    mistakes = prediction_table[prediction_table['correct'] == False].copy()
    mistakes = mistakes.sort_values('confidence', ascending=False)
    return mistakes.head(max_rows)


prediction_table = build_prediction_table(trainer, tokenized_val, val_small, label_names)
confident_mistakes = build_confident_mistakes(prediction_table, max_rows=10)

print(classification_report(
    prediction_table['true_label'],
    prediction_table['predicted_label'],
    labels=label_names,
    zero_division=0,
))

matrix = confusion_matrix(
    prediction_table['true_label'],
    prediction_table['predicted_label'],
    labels=label_names,
)

matrix_display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=label_names)
matrix_display.plot(xticks_rotation=30)
plt.title('Validation Confusion Matrix')
plt.show()

confident_mistakes


### Checks (Exercise 5)

In [ ]:
assert isinstance(prediction_table, pd.DataFrame)
for column in ['text', 'true_label', 'predicted_label', 'confidence', 'correct']:
    assert column in prediction_table.columns

assert len(prediction_table) == len(val_small)
assert prediction_table['confidence'].between(0, 1).all()
assert prediction_table['correct'].isin([True, False]).all()

assert isinstance(confident_mistakes, pd.DataFrame)
for column in ['text', 'true_label', 'predicted_label', 'confidence', 'correct']:
    assert column in confident_mistakes.columns

print('Exercise 5 passed.')


## 7. Wrap-Up Questions

1. What changed between pre-fine-tuning and post-fine-tuning predictions?
2. Which custom example was most ambiguous, and why?
3. Why is a randomly initialized classification head not useful before fine-tuning?
4. What did the attention map seem to show?
5. Why should we avoid treating one attention map as a full explanation?
6. Which confident mistake would you inspect first, and why?
